# Day 01 - SparkSession and Execution Context

**Topic:** SparkSession and Execution Context  
**Main environment:** Microsoft Fabric Notebook + Microsoft Fabric Lakehouse  
**Focus:** เข้าใจว่า PySpark application เริ่มต้นจาก `SparkSession` อย่างไร และ code ที่เราเขียนใน Notebook ทำงานอยู่ใน execution context แบบไหน

วันนี้เราจะเริ่มจากพื้นฐานสำคัญของ Spark application: `SparkSession`, `SparkContext`, Driver, Executors และการสร้าง DataFrame เบื้องต้นใน Microsoft Fabric Notebook

## Goal of the day

1. อธิบายได้ว่า `SparkSession` คือ entry point หลักของ PySpark
2. เข้าใจความต่างระหว่าง `SparkSession`, `SparkContext`, Driver และ Executors
3. ตรวจ runtime information ของ Fabric Notebook session ได้
4. สร้าง DataFrame ด้วย `spark.createDataFrame(...)` ได้
5. ใช้ `.show()`, `.count()`, `.printSchema()` เพื่อตรวจ DataFrame เบื้องต้นได้
6. Optional: เขียนและอ่าน Lakehouse table demo ได้เมื่อ attach Lakehouse แล้ว

## Why it matters in production

ใน production งาน Spark ไม่ได้ทำงานเหมือน pandas ที่ run อยู่ในเครื่องเดียวเสมอไป แต่ Spark จะมี execution model แบบ distributed

เหตุผลที่เรื่องนี้สำคัญ:

- เวลา job ช้า เราต้องรู้ว่า code ถูกรันผ่าน Driver และกระจายงานไปที่ Executors
- เวลา session หาย / notebook disconnect / Spark runtime restart เราต้องเข้าใจว่า state เช่น temp view หรือ cached DataFrame อาจหายได้
- เวลาใช้ `.show()` หรือ `.count()` เราต้องรู้ว่านี่คือ action ที่อาจ trigger Spark job
- เวลาเขียน pipeline จริง ต้องแยกให้ออกว่าอะไรอยู่ใน Notebook driver process และอะไรถูก execute แบบ distributed บน Spark runtime
- ช่วยปูพื้นให้เข้าใจ Lazy Evaluation, Job, Stage, Task และ performance debugging ในวันถัด ๆ ไป

Production takeaway:

> `SparkSession` คือประตูหลักที่เราใช้คุยกับ Spark ส่วน Driver เป็นตัวควบคุมงาน และ Executors เป็นตัวประมวลผล data partitions

## Key concepts

| Concept | Meaning |
|---|---|
| Spark | Distributed data processing engine |
| PySpark | Python API สำหรับใช้งาน Apache Spark |
| SparkSession | Entry point หลักสำหรับ DataFrame, SQL, read/write และ config |
| SparkContext | Lower-level runtime context ที่อยู่ใต้ `SparkSession` |
| Driver | Process ที่รัน notebook code, build query plan และ coordinate Spark jobs |
| Executor | Worker process ที่ประมวลผล data partitions |
| DataFrame | Distributed table-like data structure ของ Spark |
| Fabric Notebook context | Fabric เตรียม Spark runtime และ `spark` session ให้เราแล้ว |
| Action | Operation ที่ trigger execution เช่น `show`, `count`, `collect`, `write` |

## Concept explanation

### SparkSession คืออะไร?

`SparkSession` คือประตูหลักที่เราใช้คุยกับ Spark ใน PySpark

ใน Fabric Notebook ส่วนใหญ่เราไม่ต้องสร้างเอง เพราะ runtime จะเตรียมตัวแปรชื่อ `spark` มาให้แล้ว

ตัวอย่างสิ่งที่ใช้ผ่าน `spark`:

- สร้าง DataFrame: `spark.createDataFrame(...)`
- อ่าน table: `spark.read.table(...)`
- รัน SQL: `spark.sql(...)`
- เขียน table: `.saveAsTable(...)`
- เข้าถึง SparkContext: `spark.sparkContext`

### SparkContext คืออะไร?

`SparkContext` คือ object ระดับล่างกว่าที่เชื่อมไปยัง Spark runtime/cluster

ในงาน Data Engineering สมัยใหม่ เราจะใช้ `SparkSession` และ DataFrame API เป็นหลัก แต่การรู้จัก `SparkContext` ช่วยให้ inspect runtime ได้ เช่น:

- Spark application id
- Spark app name
- default parallelism
- Spark master
- Spark UI URL หรือ tracking URL ถ้า environment แสดงให้ใช้งาน

พูดง่าย ๆ:

> SparkSession = API หลักที่เราใช้ทุกวัน  
> SparkContext = runtime context ระดับล่างที่ Spark ใช้จัดการ application ภายใน

### Driver and Executors

- Driver = ตัวควบคุมหลักของ Spark application  
  ใน Fabric Notebook เราสามารถคิดแบบง่าย ๆ ว่า notebook cell ที่เรารันอยู่ฝั่ง driver

- Executors = workers ที่ Spark ใช้ประมวลผล data partitions  
  เมื่อเราเรียก action เช่น `.count()` Spark จะส่งงานไปให้ executors ประมวลผล

ถ้าคิดแบบง่าย:

> Driver = คนคุมงาน  
> Executors = คนลงมือทำงานกับ data partitions

### Fabric Notebook Execution Context

ใน Microsoft Fabric Notebook:

- `spark` มักถูกเตรียมมาให้แล้ว
- code ใน notebook cell รันผ่าน Spark session เดียวกันในช่วงที่ session ยัง active
- DataFrame, temp view, cache และ runtime state บางอย่างผูกกับ session
- ถ้า session restart สิ่งที่อยู่ใน memory/session อาจหายได้ แต่ table ที่เขียนลง Lakehouse แล้วจะยังอยู่


## Mermaid diagram: Fabric Notebook Execution Context

```mermaid
flowchart LR
    A[Fabric Notebook Cell] --> B[SparkSession: spark]
    B --> C[SparkContext]
    C --> D[Driver]
    D --> E[Executor 1]
    D --> F[Executor 2]
    D --> G[Executor N]
    E --> H[Data Partition]
    F --> I[Data Partition]
    G --> J[Data Partition]
```

Key idea:

> เราเขียน PySpark ผ่าน `spark` ใน Fabric Notebook แล้ว Spark runtime จะจัดการ execution ผ่าน Driver และ Executors

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 3, Finished, Available, Finished, False)

## Inspect SparkSession and SparkContext

In [2]:
print("Spark object:", spark)
print("Spark version:", spark.version)
print("Spark app name:", spark.sparkContext.appName)
print("Spark application id:", spark.sparkContext.applicationId)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Spark UI URL:", spark.sparkContext.uiWebUrl)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 4, Finished, Available, Finished, False)

Spark object: <pyspark.sql.session.SparkSession object at 0x7071b720eb90>
Spark version: 3.5.5.5.4.20260403.6
Spark app name: day_01_spark_session_context_dac2aba2-5163-4656-a014-3bccb62231a5
Spark application id: application_1780127094142_0001
Spark master: yarn
Default parallelism: 8
Spark UI URL: https://sparkui.fabric.microsoft.com/sparkui/7df177f2-35d4-4979-9e0b-d66832f59b88/workspaceTypes/datacloud/workspaces/a5de4c5d-a2ed-4a31-923d-ff4fcd64006e/activities/dac2aba2-5163-4656-a014-3bccb62231a5/applications/application_1780127094142_0001/jobs?capacityId=c970808a-c9ce-44a9-ae08-351ba81f0fee&pbiApi=api.fabric.microsoft.com&artifactId=ea59cba9-1d5b-42bf-ba1b-ef3b0fea25b6


## Create mock data

In [3]:
customers_data = [
    (1, "Alice", "active", "Bangkok"),
    (2, "Bob", "inactive", "Chiang Mai"),
    (3, "Charlie", "active", "Phuket"),
    (4, "Diana", "active", "Bangkok"),
    (5, "Ethan", "inactive", "Rayong"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
])

df_customers = spark.createDataFrame(customers_data, customer_schema)

df_customers.show()
df_customers.printSchema()

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 5, Finished, Available, Finished, False)

+-----------+-------------+--------+----------+
|customer_id|customer_name|  status|    region|
+-----------+-------------+--------+----------+
|          1|        Alice|  active|   Bangkok|
|          2|          Bob|inactive|Chiang Mai|
|          3|      Charlie|  active|    Phuket|
|          4|        Diana|  active|   Bangkok|
|          5|        Ethan|inactive|    Rayong|
+-----------+-------------+--------+----------+

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)



## PySpark code examples

ใน section นี้จะไล่จากการ inspect Spark runtime → สร้าง DataFrame → ใช้ action พื้นฐาน → transformation เบื้องต้น → write/read Lakehouse table

### Example 1: Inspect runtime information as a small DataFrame

บางครั้งการแปลง runtime information เป็น DataFrame เล็ก ๆ ช่วยให้ preview ง่าย และใช้ `.show()` ได้เหมือน DataFrame ปกติ

In [4]:
runtime_data = [
    ("spark_version", spark.version),
    ("app_name", spark.sparkContext.appName),
    ("application_id", spark.sparkContext.applicationId),
    ("master", spark.sparkContext.master),
    ("default_parallelism", str(spark.sparkContext.defaultParallelism)),
]

df_runtime_info = spark.createDataFrame(runtime_data, ["property", "value"])

df_runtime_info.show(truncate=False)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 6, Finished, Available, Finished, False)

+-------------------+-----------------------------------------------------------------+
|property           |value                                                            |
+-------------------+-----------------------------------------------------------------+
|spark_version      |3.5.5.5.4.20260403.6                                             |
|app_name           |day_01_spark_session_context_dac2aba2-5163-4656-a014-3bccb62231a5|
|application_id     |application_1780127094142_0001                                   |
|master             |yarn                                                             |
|default_parallelism|8                                                                |
+-------------------+-----------------------------------------------------------------+



### Example 2: Basic DataFrame actions

ตัวอย่าง action พื้นฐานที่ใช้บ่อย:

- `.show()` ใช้ preview data
- `.printSchema()` ใช้ดู schema
- `.count()` ใช้นับจำนวน rows

ทั้งสามตัวนี้ช่วยตรวจสอบ DataFrame เบื้องต้นได้ดี แต่ `.show()` และ `.count()` ก็สามารถ trigger Spark job ได้

In [5]:
df_customers.show()

df_customers.printSchema()

customer_count = df_customers.count()
print("Customer count:", customer_count)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 7, Finished, Available, Finished, False)

+-----------+-------------+--------+----------+
|customer_id|customer_name|  status|    region|
+-----------+-------------+--------+----------+
|          1|        Alice|  active|   Bangkok|
|          2|          Bob|inactive|Chiang Mai|
|          3|      Charlie|  active|    Phuket|
|          4|        Diana|  active|   Bangkok|
|          5|        Ethan|inactive|    Rayong|
+-----------+-------------+--------+----------+

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)

Customer count: 5


### Example 3: Basic DataFrame transformation

ตัวอย่างนี้ใช้ `filter` และ `select` เพื่อสร้าง DataFrame ใหม่เฉพาะ active customers

In [6]:
df_active_customers = (
    df_customers
    .filter(F.col("status") == "active")
    .select(
        "customer_id",
        "customer_name",
        "region"
    )
)

df_active_customers.show()

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 8, Finished, Available, Finished, False)

+-----------+-------------+-------+
|customer_id|customer_name| region|
+-----------+-------------+-------+
|          1|        Alice|Bangkok|
|          3|      Charlie| Phuket|
|          4|        Diana|Bangkok|
+-----------+-------------+-------+



### Example 4: Add a derived column

ใช้ `withColumn` เพื่อสร้าง column ใหม่จาก logic ง่าย ๆ

In [7]:
df_customers_with_flag = (
    df_customers
    .withColumn(
        "is_bangkok_customer",
        F.when(F.col("region") == "Bangkok", F.lit(True)).otherwise(F.lit(False))
    )
)

df_customers_with_flag.show()

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 9, Finished, Available, Finished, False)

+-----------+-------------+--------+----------+-------------------+
|customer_id|customer_name|  status|    region|is_bangkok_customer|
+-----------+-------------+--------+----------+-------------------+
|          1|        Alice|  active|   Bangkok|               true|
|          2|          Bob|inactive|Chiang Mai|              false|
|          3|      Charlie|  active|    Phuket|              false|
|          4|        Diana|  active|   Bangkok|               true|
|          5|        Ethan|inactive|    Rayong|              false|
+-----------+-------------+--------+----------+-------------------+



### Example 5: Optional Lakehouse table demo

ใน Fabric Lakehouse สามารถเขียน DataFrame เป็น table ได้ด้วย `.saveAsTable(...)`

> หมายเหตุ: ต้อง attach Lakehouse เข้ากับ Fabric Notebook ก่อนรัน cell นี้

In [8]:
table_name = "day01_customer_demo"

(
    df_customers_with_flag
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"Table written successfully: {table_name}")

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 10, Finished, Available, Finished, False)

Table written successfully: day01_customer_demo


### Example 6: Read table back from Lakehouse

หลังจากเขียน table แล้ว สามารถอ่านกลับด้วย `spark.read.table(...)`

In [9]:
df_saved_customers = spark.read.table("day01_customer_demo")

df_saved_customers.show()

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 11, Finished, Available, Finished, False)

+-----------+-------------+--------+----------+-------------------+
|customer_id|customer_name|  status|    region|is_bangkok_customer|
+-----------+-------------+--------+----------+-------------------+
|          2|          Bob|inactive|Chiang Mai|              false|
|          5|        Ethan|inactive|    Rayong|              false|
|          3|      Charlie|  active|    Phuket|              false|
|          4|        Diana|  active|   Bangkok|               true|
|          1|        Alice|  active|   Bangkok|               true|
+-----------+-------------+--------+----------+-------------------+



## Quick recap

| Question | Answer |
|---|---|
| Main PySpark entry point คืออะไร? | `SparkSession` หรือ `spark` |
| SparkContext คืออะไร? | Runtime context ระดับล่างที่อยู่ใต้ SparkSession |
| Driver ทำหน้าที่อะไร? | ควบคุม job และส่ง tasks ไปให้ executors |
| Executors ทำหน้าที่อะไร? | ประมวลผล data partitions |
| DataFrame คืออะไร? | Distributed table-like data structure |
| Basic actions มีอะไรบ้าง? | `show`, `count`, `printSchema`, `write` |

## Exercises

### Exercise 1: Inspect your Spark runtime

ให้ print ข้อมูลต่อไปนี้ออกมา:

- Spark version
- Application id
- App name
- Default parallelism
- Spark master

Expected result:

- เห็น runtime information ของ Spark session ปัจจุบัน

In [10]:
print("Spark version:", spark.version)
print("Application id:", spark.sparkContext.applicationId)
print("App name:", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Spark master:", spark.sparkContext.master)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 12, Finished, Available, Finished, False)

Spark version: 3.5.5.5.4.20260403.6
Application id: application_1780127094142_0001
App name: day_01_spark_session_context_dac2aba2-5163-4656-a014-3bccb62231a5
Default parallelism: 8
Spark master: yarn


### Exercise 2: Create a transaction DataFrame

สร้าง DataFrame ชื่อ `df_transactions` ด้วย columns:

- `transaction_id`
- `customer_id`
- `amount`
- `payment_status`

จากนั้น run:

- `.show()`
- `.printSchema()`
- `.count()`

Expected result:

- เห็น transaction data preview
- เห็น schema
- เห็น row count

In [11]:
transactions_data = [
    ("T001", 101, 1200.50, "success"),
    ("T002", 102, 0.00, "failed"),
    ("T003", 103, 850.00, "success"),
    ("T004", 104, -100.00, "success"),
    ("T005", 105, 450.75, "pending"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("payment_status", T.StringType(), True),
])

df_transactions = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions.show()
df_transactions.printSchema()

transaction_count = df_transactions.count()
print("Transaction count:", transaction_count)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 13, Finished, Available, Finished, False)

+--------------+-----------+------+--------------+
|transaction_id|customer_id|amount|payment_status|
+--------------+-----------+------+--------------+
|          T001|        101|1200.5|       success|
|          T002|        102|   0.0|        failed|
|          T003|        103| 850.0|       success|
|          T004|        104|-100.0|       success|
|          T005|        105|450.75|       pending|
+--------------+-----------+------+--------------+

root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- amount: double (nullable = true)
 |-- payment_status: string (nullable = true)

Transaction count: 5


### Exercise 3: Filter successful transactions

จาก `df_transactions` ให้สร้าง `df_success_transactions`

Requirements:

- filter เฉพาะ `payment_status = "success"`
- filter เฉพาะ `amount > 0`
- select columns:
  - `transaction_id`
  - `customer_id`
  - `amount`

Expected result:

- เห็นเฉพาะ successful transactions ที่ amount มากกว่า 0

In [12]:
df_success_transactions = (
    df_transactions
    .filter(F.col("payment_status") == "success")
    .filter(F.col("amount") > 0)
    .select(
        "transaction_id",
        "customer_id",
        "amount"
    )
)

df_success_transactions.show()

success_count = df_success_transactions.count()
print("Successful transaction count:", success_count)

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 14, Finished, Available, Finished, False)

+--------------+-----------+------+
|transaction_id|customer_id|amount|
+--------------+-----------+------+
|          T001|        101|1200.5|
|          T003|        103| 850.0|
+--------------+-----------+------+

Successful transaction count: 2


## Common mistakes

1. **พยายามสร้าง SparkSession ใหม่ใน Fabric โดยไม่จำเป็น**

   ใน Fabric Notebook มักมี `spark` พร้อมใช้อยู่แล้ว การสร้างใหม่ซ้ำอาจทำให้งงเรื่อง session/state

2. **คิดว่า Spark DataFrame = pandas DataFrame**

   Spark DataFrame เป็น distributed DataFrame ไม่ได้อยู่ใน memory ของ driver ทั้งหมดแบบ pandas

3. **ใช้ `collect()` โดยไม่ระวัง**

   `collect()` จะดึงข้อมูลทั้งหมดจาก executor กลับเข้า driver ถ้าข้อมูลใหญ่เกินไป driver อาจ memory เต็มหรือ notebook ค้างได้
   - ใช้ได้เมื่อมั่นใจว่าข้อมูลมีขนาดเล็กมาก เช่น debug sample ไม่กี่ rows หรือผลลัพธ์ aggregate เล็ก ๆ
   - ถ้าต้องการดูข้อมูลเฉย ๆ ควรใช้ `.show()`, `.limit()`, หรือ `.sample()` แทน
   - ก่อนใช้ `collect()` ควร `select`, `filter`, หรือ `limit` ให้ข้อมูลเล็กลงก่อน

4. **ไม่ดู schema ก่อน transform**

   ถ้าไม่ตรวจ schema อาจ cast ผิด type หรือ filter ผิด logic ได้

5. **เขียน table โดยไม่ได้ attach Lakehouse**

   `saveAsTable()` อาจ fail ถ้า Notebook ยังไม่ได้ attach Lakehouse หรือ user ที่รัน notebook ไม่มี permission เพียงพอ


## Expected Output / Success Criteria

เมื่อจบ Day 01 ควรทำได้:

- เห็น Spark version และ application id ของ notebook session
- อธิบายได้ว่า `SparkSession` คือ entry point หลักของ PySpark
- อธิบายได้ว่า `SparkContext` เป็น lower-level runtime context
- อธิบายแบบง่าย ๆ ได้ว่า Driver ควบคุมงาน ส่วน Executors ประมวลผล data partitions
- สร้าง DataFrame ด้วย `spark.createDataFrame(...)` ได้
- ใช้ `.show()`, `.count()`, `.printSchema()` ได้
- สร้าง transformation ง่าย ๆ ด้วย `filter`, `select`, `withColumn` ได้
- รู้ว่า `collect()` ไม่ควรใช้กับข้อมูลใหญ่ เพราะดึงข้อมูลทั้งหมดกลับมาที่ driver

## Final takeaway

Day 01 คือการเข้าใจว่า PySpark code ใน Fabric Notebook ไม่ได้ทำงานแบบ Python ธรรมดาบนเครื่องเดียว แต่ทำงานผ่าน Spark runtime

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- `spark` หรือ `SparkSession` คือ entry point หลักของ PySpark
- `SparkContext` คือ runtime context ระดับล่างที่ Spark ใช้ภายใน
- Driver ควบคุมงาน และ Executors ประมวลผล data partitions
- Spark DataFrame เป็น distributed data structure ไม่ใช่ pandas DataFrame
- `.collect()` ต้องใช้อย่างระวัง เพราะดึงข้อมูลทั้งหมดกลับมาที่ driver

## Optional cleanup

In [13]:
# spark.sql("DROP TABLE IF EXISTS day01_customer_demo")

StatementMeta(, dac2aba2-5163-4656-a014-3bccb62231a5, 15, Finished, Available, Finished, False)